# 05. Evaluación en Test

Evalúa los modelos **ya entrenados** contra el **test set**: los chips que nunca se usaron ni en entrenamiento ni en validación.

Busca checkpoints (`*_best.pt`) en `data/06_models/` **y sus subcarpetas**, así que sirve sin importar de cuál notebook de entrenamiento vengan:

| Carpeta | Notebook de origen |
|---|---|
| `data/06_models/*_best.pt` | `04_training_models.ipynb` (preentrenado, original) |
| `data/06_models/scratch/*_best.pt` | `04b_training_models_scratch.ipynb` (sin preentrenar) |
| `data/06_models/tuned/*_best.pt` | `04c_training_models_tuned.ipynb` (preentrenado, LR/paciencia ajustados) |

Esto da el número "real" de qué tan bien funciona cada modelo/variante con datos nuevos.

Resultados en texto/CSV → `data/08_reporting/`

## 1. Configuración

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import confusion_matrix

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.data.dataset import create_dataloaders
from src.models.train import create_model
from src.utils.helpers import get_device, set_seed
from src.utils.metrics import compute_metrics

DATA_DIR = ROOT / "data"
MODELS_DIR = DATA_DIR / "06_models"
FIGURES_DIR = ROOT / "reports" / "figures"
REPORT_DIR = DATA_DIR / "08_reporting"

BATCH_SIZE = 32
NUM_WORKERS = 4
SEED = 42
LABELS = ["sem_garimpo", "com_garimpo"]

REPORT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
device = get_device()
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else str(device)

print(f"Device: {device}")
print(f"GPU: {gpu_name}")
print(f"PyTorch: {torch.__version__}")

## 2. Test DataLoader

Son los chips reservados en `02_preprocessing.ipynb` que ningún modelo vio durante `04_training_models.ipynb`.

In [ ]:
loaders = create_dataloaders(
    data_dir=DATA_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)
test_loader = loaders["test"]

print(f"Test: {len(test_loader.dataset):,} chips | {len(test_loader):,} batches")

## 3. Checkpoints disponibles

Busca automáticamente todos los `*_best.pt` en `data/06_models/` **incluyendo subcarpetas** (`scratch/`, `tuned/`, etc.). No importa cuántos haya: cada uno se evalúa por separado, y se identifica por su "variante" (la subcarpeta donde está, o "original" si está directo en `data/06_models/`).

In [ ]:
# Si algún .pt no se detecta solo (nombre/ruta distinta, quedó en otro lado, etc.),
# agrega aquí su ruta relativa a data/06_models/ (puede incluir subcarpeta):
MANUAL_CHECKPOINTS = [
    # "resnet50_best.pt",
    # "tuned/resnet50_best.pt",
    # "scratch/resnet50_best.pt",
]


def get_variant(ckpt_path: Path, models_dir: Path) -> str:
    """'original' si el .pt está directo en models_dir, o el nombre de la subcarpeta."""
    rel_parent = ckpt_path.relative_to(models_dir).parent
    return "original" if str(rel_parent) == "." else str(rel_parent)


checkpoint_paths = sorted(MODELS_DIR.rglob("*_best.pt"))
for name in MANUAL_CHECKPOINTS:
    path = MODELS_DIR / name
    if path not in checkpoint_paths:
        checkpoint_paths.append(path)

print(f"Checkpoints encontrados: {len(checkpoint_paths)}")
for p in checkpoint_paths:
    variant = get_variant(p, MODELS_DIR) if p.exists() else "?"
    print(f"  - [{variant}] {p.name}" + ("  (⚠️ no existe)" if not p.exists() else ""))

if not checkpoint_paths:
    print(
        "\n⚠️  No hay checkpoints en data/06_models/ (ni en sus subcarpetas).\n"
        "Corre 04_training_models.ipynb primero, o copia ahí los .pt ya entrenados."
    )

## 4. Helpers de evaluación

In [ ]:
@torch.no_grad()
def evaluate_checkpoint(ckpt_path: Path, loader, device) -> dict:
    """Carga un checkpoint y corre inferencia completa sobre `loader`."""
    ckpt = torch.load(ckpt_path, map_location=device)
    model_name = ckpt["model_name"]
    img_size = ckpt.get("img_size", 128)

    model = create_model(model_name, img_size=img_size, pretrained=False).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    all_preds, all_labels = [], []
    for images, labels in loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.tolist())

    metrics = compute_metrics(all_labels, all_preds)
    metrics.update(
        {
            "model_name": model_name,
            "train_best_epoch": ckpt.get("epoch"),
            "y_true": all_labels,
            "y_pred": all_preds,
        }
    )
    return metrics


def plot_confusion_matrix(y_true, y_pred, title: str, save_path: Path) -> None:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4, 4))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=LABELS, yticklabels=LABELS, ax=ax,
    )
    ax.set_xlabel("Predicho")
    ax.set_ylabel("Real")
    ax.set_title(title)
    plt.tight_layout()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()

## 5. Evaluar cada checkpoint en test

In [ ]:
results = []
log_lines = [
    "EVALUACIÓN EN TEST — RESULTADOS",
    f"Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M')}",
    f"Device: {device} | GPU: {gpu_name}",
    f"Test chips: {len(test_loader.dataset):,}",
    "",
]

for i, ckpt_path in enumerate(checkpoint_paths, start=1):
    variant = get_variant(ckpt_path, MODELS_DIR) if ckpt_path.exists() else "?"

    print("\n" + "=" * 60)
    print(f"[{i}/{len(checkpoint_paths)}] Evaluando: [{variant}] {ckpt_path.name}")
    print("=" * 60)

    if not ckpt_path.exists():
        print(f"⚠️  Saltado: no existe {ckpt_path}")
        continue

    m = evaluate_checkpoint(ckpt_path, test_loader, device)
    label = f"{m['model_name']} [{variant}]"

    print(
        f"accuracy {m['accuracy']:.4f} | macro F1 {m['macro_f1']:.4f} | "
        f"recall com_garimpo {m['recall_com_garimpo']:.4f} | "
        f"precision com_garimpo {m['precision_com_garimpo']:.4f}"
    )

    fig_name = f"{ckpt_path.stem.replace('_best', '')}_{variant}"
    plot_confusion_matrix(
        m["y_true"], m["y_pred"],
        title=f"{label} — Test",
        save_path=FIGURES_DIR / f"{fig_name}_test_confusion.png",
    )

    log_lines += [
        f"MODELO: {label} ({ckpt_path.name})",
        f"  test accuracy: {m['accuracy']:.4f}",
        f"  test macro F1: {m['macro_f1']:.4f}",
        f"  f1 com_garimpo: {m['f1_com_garimpo']:.4f} | f1 sem_garimpo: {m['f1_sem_garimpo']:.4f}",
        f"  recall com_garimpo: {m['recall_com_garimpo']:.4f} | precision com_garimpo: {m['precision_com_garimpo']:.4f}",
        "",
    ]

    results.append({
        "model": m["model_name"],
        "variant": variant,
        "checkpoint": ckpt_path.name,
        "train_best_epoch": m["train_best_epoch"],
        "test_accuracy": round(m["accuracy"], 4),
        "test_macro_f1": round(m["macro_f1"], 4),
        "test_f1_com_garimpo": round(m["f1_com_garimpo"], 4),
        "test_f1_sem_garimpo": round(m["f1_sem_garimpo"], 4),
        "test_recall_com_garimpo": round(m["recall_com_garimpo"], 4),
        "test_precision_com_garimpo": round(m["precision_com_garimpo"], 4),
    })

print("\nEvaluación completada.")

## 6. Resumen comparativo (¿qué tan bien funciona cada modelo de verdad?)

In [ ]:
results_df = pd.DataFrame(results).sort_values("test_macro_f1", ascending=False)
display(results_df)

summary_csv = REPORT_DIR / "test_results_summary.csv"
results_df.to_csv(summary_csv, index=False)

best = results_df.iloc[0]
log_lines += [
    "=" * 60,
    "RESUMEN COMPARATIVO (test)",
    "=" * 60,
    "",
    results_df.to_string(index=False),
    "",
    f"MEJOR MODELO EN TEST: {best['model']} [{best['variant']}] | "
    f"test macro F1 = {best['test_macro_f1']} | accuracy = {best['test_accuracy']}",
    f"Checkpoint: {best['checkpoint']}",
]

summary_txt = REPORT_DIR / "test_results.txt"
summary_txt.write_text("\n".join(log_lines), encoding="utf-8")

print(f"\nArchivos guardados:")
print(f"  {summary_csv}")
print(f"  {summary_txt}")
print(f"\nMejor modelo en TEST: {best['model']} [{best['variant']}]")
print(f"  Accuracy:  {best['test_accuracy']*100:.1f}%")
print(f"  Macro F1:  {best['test_macro_f1']*100:.1f}%")

## 7. Comparar val vs test (¿algún modelo se cae al pasar a datos nuevos?)

Como val y test son rasters distintos, comparar sus métricas ayuda a detectar sobreajuste a rasters específicos.

In [ ]:
# Cada "variante" de entrenamiento guarda su propio resumen de validación.
# Se cruza cada checkpoint con el CSV de val que le corresponde según su variante.
VAL_SUMMARY_BY_VARIANT = {
    "original": REPORT_DIR / "training_results_summary.csv",
    "tuned": REPORT_DIR / "training_results_summary_tuned.csv",
    "scratch": REPORT_DIR / "training_results_summary_scratch.csv",
}

val_frames = []
for variant, path in VAL_SUMMARY_BY_VARIANT.items():
    if path.exists():
        vdf = pd.read_csv(path)[["timm_name", "val_macro_f1", "val_accuracy"]]
        vdf["variant"] = variant
        val_frames.append(vdf)

if val_frames:
    val_df = pd.concat(val_frames, ignore_index=True)
    comparison = results_df.merge(
        val_df, left_on=["model", "variant"], right_on=["timm_name", "variant"], how="left",
    ).drop(columns=["timm_name"])
    comparison["f1_drop_val_to_test"] = (
        comparison["val_macro_f1"] - comparison["test_macro_f1"]
    ).round(4)

    cols = ["model", "variant", "val_macro_f1", "test_macro_f1", "f1_drop_val_to_test",
            "val_accuracy", "test_accuracy"]
    display(comparison[cols].sort_values("f1_drop_val_to_test"))

    comparison_csv = REPORT_DIR / "val_vs_test_comparison.csv"
    comparison[cols].to_csv(comparison_csv, index=False)
    print(f"\nGuardado: {comparison_csv}")
else:
    print(
        "No se encontró ningún CSV de validación "
        "(training_results_summary[_tuned/_scratch].csv) para comparar."
    )

## 8. Próximo paso

Con el mejor modelo identificado (mayor `test_macro_f1`), ya tienes evidencia lista para el informe: `data/08_reporting/test_results_summary.csv`, `test_results.txt`, `val_vs_test_comparison.csv`, y las matrices de confusión en `reports/figures/*_test_confusion.png`.